In [1]:
# Data Preprocessing 
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = r"C:\Users\alifa\OneDrive\Desktop\dissertation_project\data\MachineLearningCVE"
SAVE_PATH = r"C:\Users\alifa\OneDrive\Desktop\dissertation_project\data"

print("Loading data...")
csv_files = [f for f in os.listdir(DATA_PATH) if f.endswith('.csv')]
dfs = []
for f in csv_files:
    df_temp = pd.read_csv(os.path.join(DATA_PATH, f), encoding='utf-8', low_memory=False)
    df_temp.columns = df_temp.columns.str.strip()
    dfs.append(df_temp)
df = pd.concat(dfs, ignore_index=True)
print(f"Loaded: {len(df):,} rows x {len(df.columns)} columns")

# Drop duplicate rows
before = len(df)
df = df.drop_duplicates()
print(f"\nStep 1 — Duplicates removed: {before - len(df):,}")
print(f"  Remaining rows: {len(df):,}")

Loading data...
Loaded: 2,830,743 rows x 79 columns

Step 1 — Duplicates removed: 308,381
  Remaining rows: 2,522,362


In [2]:
# Handle Infinite & Missing Values 

# Replace infinite values with NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)
print("Step 2 — Replaced infinite values with NaN")

# Drop rows with any remaining NaN values
before = len(df)
df = df.dropna()
print(f"Step 3 — Rows dropped due to NaN: {before - len(df):,}")
print(f"  Remaining rows: {len(df):,}")

# Verify no more nulls or infinities
assert df.isnull().sum().sum() == 0, "Still has null values!"
assert not np.isinf(df.select_dtypes(include=[np.number]).values).any(), "Still has infinite values!"
print("\nVerification passed — no nulls or infinite values remaining")

Step 2 — Replaced infinite values with NaN
Step 3 — Rows dropped due to NaN: 1,564
  Remaining rows: 2,520,798

Verification passed — no nulls or infinite values remaining


In [3]:
# Remove Problematic Columns 
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Drop zero-variance columns (all same value)
zero_var_cols = [col for col in numeric_cols if df[col].nunique() <= 1]
df = df.drop(columns=zero_var_cols)
print(f"Step 4 — Zero-variance columns dropped: {len(zero_var_cols)}")
print(f"  Dropped: {zero_var_cols}")

# Drop columns that are all zeros (bulk rate columns)
all_zero_cols = [col for col in df.select_dtypes(include=[np.number]).columns
                 if df[col].sum() == 0]
df = df.drop(columns=all_zero_cols)
print(f"\nStep 5 — All-zero columns dropped: {len(all_zero_cols)}")
print(f"  Dropped: {all_zero_cols}")

print(f"\nRemaining features: {len(df.select_dtypes(include=[np.number]).columns)}")
print(f"Remaining rows: {len(df):,}")

Step 4 — Zero-variance columns dropped: 8
  Dropped: ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']

Step 5 — All-zero columns dropped: 0
  Dropped: []

Remaining features: 70
Remaining rows: 2,520,798


In [4]:
# Encode Labels & Save 
from sklearn.preprocessing import LabelEncoder

# Encode string labels to integers
le = LabelEncoder()
df['Label_encoded'] = le.fit_transform(df['Label'])

# Show label mapping
print("Label encoding mapping:")
for i, label in enumerate(le.classes_):
    count = (df['Label_encoded'] == i).sum()
    print(f"  {i:2d} → {label:<40} ({count:,} records)")

# Save clean dataset
save_file = os.path.join(SAVE_PATH, 'cicids2017_clean.csv')
df.to_csv(save_file, index=False)
print(f"\nClean dataset saved to: {save_file}")
print(f"Final shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

Label encoding mapping:
   0 → BENIGN                                   (2,095,057 records)
   1 → Bot                                      (1,948 records)
   2 → DDoS                                     (128,014 records)
   3 → DoS GoldenEye                            (10,286 records)
   4 → DoS Hulk                                 (172,846 records)
   5 → DoS Slowhttptest                         (5,228 records)
   6 → DoS slowloris                            (5,385 records)
   7 → FTP-Patator                              (5,931 records)
   8 → Heartbleed                               (11 records)
   9 → Infiltration                             (36 records)
  10 → PortScan                                 (90,694 records)
  11 → SSH-Patator                              (3,219 records)
  12 → Web Attack � Brute Force                 (1,470 records)
  13 → Web Attack � Sql Injection               (21 records)
  14 → Web Attack � XSS                         (652 records)

Clean dataset sa

In [5]:
# Feature Selection 
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns

# Separate features and label
feature_cols = [col for col in df.columns if col not in ['Label', 'Label_encoded']]
X = df[feature_cols].select_dtypes(include=[np.number])
y = df['Label_encoded']

print(f"Features before selection: {X.shape[1]}")
print(f"Samples: {len(X):,}")

# Remove highly correlated features (threshold > 0.95) 
corr_matrix = X.corr().abs()
upper_tri = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)
high_corr_cols = [col for col in upper_tri.columns if any(upper_tri[col] > 0.95)]
print(f"\nHighly correlated features removed (>0.95): {len(high_corr_cols)}")
print(f"  Removed: {high_corr_cols}")

X = X.drop(columns=high_corr_cols)
print(f"\nFeatures after correlation removal: {X.shape[1]}")

Features before selection: 70
Samples: 2,520,798

Highly correlated features removed (>0.95): 23
  Removed: ['Total Backward Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Std', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Fwd IAT Total', 'Fwd IAT Max', 'Fwd Packets/s', 'Packet Length Std', 'SYN Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Average Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size', 'Fwd Header Length.1', 'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Subflow Bwd Packets', 'Subflow Bwd Bytes', 'Idle Mean', 'Idle Max', 'Idle Min']

Features after correlation removal: 47


In [6]:
# Normalisation 
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns
)

print("Normalisation complete using MinMaxScaler")
print(f"Feature value range after scaling:")
print(f"  Min: {X_scaled.min().min():.4f}")
print(f"  Max: {X_scaled.max().max():.4f}")
print(f"  All values between 0 and 1: {((X_scaled >= 0) & (X_scaled <= 1)).all().all()}")

Normalisation complete using MinMaxScaler
Feature value range after scaling:
  Min: 0.0000
  Max: 1.0000
  All values between 0 and 1: False


In [7]:
# Save Final Feature Set 
import joblib

SAVE_PATH = r"C:\Users\alifa\OneDrive\Desktop\dissertation_project\data"

# Combine scaled features with labels
df_final = X_scaled.copy()
df_final['Label'] = df['Label'].values
df_final['Label_encoded'] = y.values

# Save final dataset
final_file = os.path.join(SAVE_PATH, 'cicids2017_final.csv')
df_final.to_csv(final_file, index=False)

# Save scaler and label encoder for later use
joblib.dump(scaler, os.path.join(SAVE_PATH, 'scaler.pkl'))
joblib.dump(le, os.path.join(SAVE_PATH, 'label_encoder.pkl'))

# Save feature names for XAI phase
feature_names = X_scaled.columns.tolist()
joblib.dump(feature_names, os.path.join(SAVE_PATH, 'feature_names.pkl'))

print(f"Final dataset saved: {final_file}")
print(f"Scaler saved: scaler.pkl")
print(f"Label encoder saved: label_encoder.pkl")
print(f"Feature names saved: feature_names.pkl")
print(f"\nFinal dataset shape: {df_final.shape[0]:,} rows x {df_final.shape[1]} columns")
print(f"Features used for modelling: {len(feature_names)}")
print(f"\nFeature names:")
for i, f in enumerate(feature_names):
    print(f"  {i+1:2d}. {f}")

Final dataset saved: C:\Users\alifa\OneDrive\Desktop\dissertation_project\data\cicids2017_final.csv
Scaler saved: scaler.pkl
Label encoder saved: label_encoder.pkl
Feature names saved: feature_names.pkl

Final dataset shape: 2,520,798 rows x 49 columns
Features used for modelling: 47

Feature names:
   1. Destination Port
   2. Flow Duration
   3. Total Fwd Packets
   4. Total Length of Fwd Packets
   5. Fwd Packet Length Max
   6. Fwd Packet Length Min
   7. Fwd Packet Length Mean
   8. Bwd Packet Length Max
   9. Bwd Packet Length Min
  10. Flow Bytes/s
  11. Flow Packets/s
  12. Flow IAT Mean
  13. Flow IAT Std
  14. Flow IAT Max
  15. Flow IAT Min
  16. Fwd IAT Mean
  17. Fwd IAT Std
  18. Fwd IAT Min
  19. Bwd IAT Total
  20. Bwd IAT Mean
  21. Bwd IAT Std
  22. Bwd IAT Max
  23. Bwd IAT Min
  24. Fwd PSH Flags
  25. Fwd URG Flags
  26. Fwd Header Length
  27. Bwd Header Length
  28. Bwd Packets/s
  29. Min Packet Length
  30. Max Packet Length
  31. Packet Length Mean
  32. Packe

In [2]:
print ("check")

check
